# 🏃 Nike PH Product Scraper
### Relu Consultancy — Data Extraction Engineer Hiring Challenge

---

**Author:** Somesh Shukla  
**Email:** someshshukla263@gmail.com  
**Phone:** 9179682083  
**Projects:** [shikshasync.in](https://shikshasync.in) · [dharmsetu.org.in](https://dharmsetu.org.in)

---

## 📋 What This Notebook Does

Scrapes **all products** from [https://www.nike.com/ph/w](https://www.nike.com/ph/w) and extracts **14 fields** per product:

| # | Field | Source |
|---|-------|--------|
| 1 | `Product_URL` | Nike product wall API |
| 2 | `Product_Image_URL` | Nike product wall API |
| 3 | `Product_Tagging` | `badgeLabel` from API |
| 4 | `Product_Name` | `copy.title + subTitle` |
| 5 | `Product_Description` | JSON-LD on PDP page |
| 6 | `Original_Price` | `prices.initialPrice` |
| 7 | `Discount_Price` | `prices.currentPrice` |
| 8 | `Sizes_Available` | Size selectors on PDP |
| 9 | `Vouchers` | N/A on Nike PH |
| 10 | `Available_Colors` | Colorway links on PDP |
| 11 | `Color_Shown` | `displayColors.colorDescription` |
| 12 | `Style_Code` | `productCode` |
| 13 | `Rating_Score` | JSON-LD `aggregateRating` |
| 14 | `Review_Count` | JSON-LD `aggregateRating` |

## 🔄 3-Phase Strategy

- **Phase 1** — Hit Nike's internal `product_wall` API (paginated, 100/page) to enumerate every product colorway — no browser needed.
- **Phase 2** — Open each unique Product Detail Page (PDP) with Playwright (headless Chromium, 15 concurrent) and pull details from embedded JSON-LD.
- **Phase 3** — Merge data, write CSVs, print analytics.

## 📁 Outputs

| File | Contents |
|------|----------|
| `nike_products.csv` | Products with **non-empty Tagging AND non-empty Discount Price** |
| `top_20_rating_review.csv` | Top 20 by Rating+Reviews where Review Count > 150 |
| Console | Count of empty-tagging products + Top 10 most expensive |

---
## ⚙️ Step 0 — Install Dependencies

Install Playwright and download the Chromium browser binary. This takes ~1-2 minutes on first run.

In [ ]:
# Install required packages
!pip install -q playwright

# Install Chromium browser for Playwright
!playwright install chromium
!playwright install-deps chromium

print('✅ Dependencies installed successfully!')

---
## 📦 Step 1 — Imports & Configuration

In [ ]:
from __future__ import annotations

import asyncio
import csv
import json
import os
import re
import time
from urllib.request import Request, urlopen

from playwright.async_api import async_playwright

# ─── API endpoint (paginated, 100 products per page) ───
LISTING_API = (
    "https://api.nike.com/discover/product_wall/v1/marketplace/PH/"
    "language/en-GB/consumerChannelId/d9a5bc42-4b9c-4976-858a-f159cf99c647"
    "?path=/ph/w&queryType=PRODUCTS&anchor={anchor}&count=100"
)

NIKE_API_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
    ),
    "nike-api-caller-id": "nike:dotcom:browse:wall.client:2.0",
    "Accept": "application/json",
    "Accept-Language": "en-GB,en;q=0.9",
}

CSV_HEADERS = [
    "Product_URL", "Product_Image_URL", "Product_Tagging", "Product_Name",
    "Product_Description", "Original_Price", "Discount_Price",
    "Sizes_Available", "Vouchers", "Available_Colors", "Color_Shown",
    "Style_Code", "Rating_Score", "Review_Count",
]

# Cache files (avoid re-scraping on re-run)
LISTING_CACHE  = "listing.json"
DETAILS_CACHE  = "details.json"
MAIN_CSV       = "nike_products.csv"
TOP20_CSV      = "top_20_rating_review.csv"

# Concurrency for PDP phase
CONCURRENCY    = 8   # lower than local to be Colab-friendly
NAV_TIMEOUT_MS = 30_000

print('✅ Configuration loaded.')

---
## 🔍 Phase 1 — Enumerate Products via Nike API

Nike's listing page uses a virtual scroll that only keeps ~240 DOM nodes visible at a time.  
Instead we hit Nike's **internal `product_wall` REST API** directly — clean JSON, no JS rendering needed.  
Each page returns up to 100 products; we paginate using the `anchor` parameter until `next` is absent.

In [ ]:
def fetch_listing() -> list[dict]:
    """Page through the Nike product_wall API and collect every product colorway."""
    products: list[dict] = []
    anchor = 0
    total = None

    while True:
        req = Request(LISTING_API.format(anchor=anchor), headers=NIKE_API_HEADERS)
        with urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))

        if total is None:
            total = data.get("pages", {}).get("totalResources")
            print(f"[listing] Total products on Nike PH: {total}")

        for grouping in data.get("productGroupings", []) or []:
            for p in grouping.get("products", []) or []:
                products.append(p)

        next_path = data.get("pages", {}).get("next")
        print(f"  anchor={anchor:>4d}  →  collected so far: {len(products)}")

        if not next_path:
            break
        m = re.search(r"anchor=(\d+)", next_path)
        if not m:
            break
        anchor = int(m.group(1))
        time.sleep(0.4)   # gentle pacing

    return products


def listing_to_card(p: dict) -> dict:
    """Flatten a raw API product entry into a tidy card dict."""
    prices = p.get("prices") or {}
    initial = prices.get("initialPrice")
    current = prices.get("currentPrice")
    has_discount = (
        initial is not None
        and current is not None
        and initial != current
    )
    return {
        "product_url"    : (p.get("pdpUrl") or {}).get("url", ""),
        "image_url"      : (
            (p.get("colorwayImages") or {}).get("squarishURL")
            or (p.get("colorwayImages") or {}).get("portraitURL")
            or ""
        ),
        "tagging"        : p.get("badgeLabel") or "",
        "title"          : (p.get("copy") or {}).get("title", ""),
        "subtitle"       : (p.get("copy") or {}).get("subTitle", ""),
        "color_shown"    : (p.get("displayColors") or {}).get("colorDescription") or "",
        "style_code"     : p.get("productCode") or "",
        "currency"       : prices.get("currency") or "PHP",
        "original_price" : initial if has_discount else current,
        "discount_price" : current if has_discount else None,
        "group_key"      : p.get("groupKey") or "",
    }

print('✅ Phase 1 functions defined.')

In [ ]:
# ── Run Phase 1 (uses cache if available) ──
if os.path.exists(LISTING_CACHE):
    print(f"[listing] Loading cached {LISTING_CACHE} …")
    with open(LISTING_CACHE, encoding="utf-8") as f:
        cards = json.load(f)
    print(f"[listing] {len(cards)} products loaded from cache.")
else:
    print("[listing] Fetching product list from Nike API …")
    raw   = fetch_listing()
    cards = [listing_to_card(p) for p in raw]

    # Deduplicate by URL (defensive)
    seen, unique = set(), []
    for c in cards:
        if c["product_url"] and c["product_url"] not in seen:
            seen.add(c["product_url"])
            unique.append(c)
    cards = unique

    with open(LISTING_CACHE, "w", encoding="utf-8") as f:
        json.dump(cards, f, ensure_ascii=False, indent=2)
    print(f"[listing] ✅ {len(cards)} unique products saved to {LISTING_CACHE}")

print(f"\nTotal product colorways: {len(cards)}")

---
## 🎭 Phase 2 — Scrape Product Detail Pages (Playwright)

For each **unique group key** (product family), we open the PDP with headless Chromium and extract:
- **Description** from `<script type="application/ld+json">` (JSON-LD `ProductGroup`)
- **Rating & Review Count** from `aggregateRating` in JSON-LD
- **Available Sizes** from `[data-testid="pdp-grid-selector-item"]`
- **Available Colors** count from `[data-testid^="colorway-link-"]`

Images, fonts, media and stylesheets are **blocked** for speed.  
Progress is **saved every 50 PDPs** — safe to restart if Colab disconnects.

In [ ]:
async def fetch_pdp(page, url: str) -> dict:
    """Extract description, rating, sizes and colorway count from one PDP."""
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=NAV_TIMEOUT_MS)
        try:
            await page.wait_for_selector('script[type="application/ld+json"]', timeout=8000)
        except Exception:
            pass

        return await page.evaluate(
            r"""
            () => {
              const out = {};
              const lds = [...document.querySelectorAll('script[type="application/ld+json"]')];
              let prod = null;
              for (const s of lds) {
                try {
                  const j = JSON.parse(s.innerText);
                  const arr = Array.isArray(j) ? j : [j];
                  for (const x of arr) {
                    if (x && (x['@type'] === 'ProductGroup' || x['@type'] === 'Product')) {
                      prod = x; break;
                    }
                  }
                } catch (e) {}
                if (prod) break;
              }
              if (prod) {
                out.description = prod.description || '';
                if (prod.aggregateRating) {
                  out.rating = prod.aggregateRating.ratingValue;
                  out.reviewCount = prod.aggregateRating.reviewCount;
                }
              }
              out.sizes = [...document.querySelectorAll('[data-testid="pdp-grid-selector-item"]')]
                .map(e => (e.innerText || '').trim()).filter(Boolean);
              out.colorways = document.querySelectorAll('[data-testid^="colorway-link-"]').length;
              return out;
            }
            """
        )
    except Exception as e:
        return {"_error": str(e)}


async def run_detail_phase(cards: list[dict], details: dict) -> dict:
    """Visit one PDP per unique groupKey concurrently. Results are cached."""
    seen_groups: set[str] = set()
    targets: list[dict] = []

    for c in cards:
        gk = c.get("group_key") or c["product_url"]
        if gk in seen_groups or gk in details:
            seen_groups.add(gk)
            continue
        if not c["product_url"]:
            continue
        seen_groups.add(gk)
        targets.append(c)

    print(f"[detail] {len(targets)} unique PDPs to fetch  (concurrency={CONCURRENCY})")
    if not targets:
        return details

    sem = asyncio.Semaphore(CONCURRENCY)
    done_count = 0
    save_every = 50

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(
            user_agent=NIKE_API_HEADERS["User-Agent"],
            locale="en-PH",
            viewport={"width": 1280, "height": 900},
        )
        # Block heavy assets for speed
        await ctx.route(
            "**/*",
            lambda route: route.abort()
            if route.request.resource_type in {"image", "media", "font", "stylesheet"}
            else route.continue_(),
        )

        async def worker(card):
            nonlocal done_count
            async with sem:
                page = await ctx.new_page()
                try:
                    d = await fetch_pdp(page, card["product_url"])
                finally:
                    await page.close()
            details[card.get("group_key") or card["product_url"]] = d
            done_count += 1
            if done_count % save_every == 0:
                with open(DETAILS_CACHE, "w", encoding="utf-8") as f:
                    json.dump(details, f, ensure_ascii=False)
                print(f"  [detail] {done_count}/{len(targets)} saved …")

        await asyncio.gather(*[worker(c) for c in targets])
        await ctx.close()
        await browser.close()

    with open(DETAILS_CACHE, "w", encoding="utf-8") as f:
        json.dump(details, f, ensure_ascii=False)
    print(f"[detail] ✅ Done: {done_count} PDPs fetched.")
    return details

print('✅ Phase 2 functions defined.')

In [ ]:
# ── Run Phase 2 (resumes from cache automatically) ──
details: dict = {}
if os.path.exists(DETAILS_CACHE):
    with open(DETAILS_CACHE, encoding="utf-8") as f:
        details = json.load(f)
    print(f"[detail] Resuming: {len(details)} PDPs already cached.")

details = await run_detail_phase(cards, details)

---
## 📊 Phase 3 — Merge, Analyse & Write CSVs

- Merges listing cards with PDP details
- **`nike_products.csv`** — only rows with non-empty `Product_Tagging` **AND** non-empty `Discount_Price`
- **Console** — count of empty-tagging products + Top 10 most expensive by Discount Price
- **`top_20_rating_review.csv`** — Top 20 ranked by Rating (desc) then Review Count (desc), where Review Count > 150

In [ ]:
def merge(card: dict, detail: dict) -> dict:
    """Combine a listing card with its PDP detail into the final output row."""
    detail = detail or {}
    sizes  = detail.get("sizes") or []
    return {
        "Product_URL"         : card["product_url"],
        "Product_Image_URL"   : card["image_url"],
        "Product_Tagging"     : card["tagging"],
        "Product_Name"        : (card["title"] + " " + card["subtitle"]).strip(),
        "Product_Description" : detail.get("description") or "",
        "Original_Price"      : card["original_price"] if card["original_price"] is not None else "",
        "Discount_Price"      : card["discount_price"] if card["discount_price"] is not None else "",
        "Sizes_Available"     : ", ".join(sizes),
        "Vouchers"            : "",   # Nike PH does not surface voucher offers
        "Available_Colors"    : detail.get("colorways") or "",
        "Color_Shown"         : card["color_shown"],
        "Style_Code"          : card["style_code"],
        "Rating_Score"        : detail.get("rating")       if detail.get("rating")       is not None else "",
        "Review_Count"        : detail.get("reviewCount")  if detail.get("reviewCount")  is not None else "",
    }


def write_outputs(rows: list[dict]) -> None:
    # ── Step 3: Count empty-tagging products ──
    empty_tagging = sum(1 for r in rows if not str(r["Product_Tagging"]).strip())
    print(f"\n{'='*60}")
    print(f"  Total products with empty tagging: {empty_tagging}")
    print(f"{'='*60}\n")

    # ── Step 4: Save nike_products.csv (tagged + discounted only) ──
    main_rows = [
        r for r in rows
        if str(r["Product_Tagging"]).strip() and str(r["Discount_Price"]).strip()
    ]
    with open(MAIN_CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_HEADERS)
        w.writeheader()
        w.writerows(main_rows)
    print(f"[csv] ✅ {len(main_rows)} rows → {MAIN_CSV}")

    # ── Step 5A: Top 10 most expensive (by Discount_Price) ──
    priced = []
    for r in main_rows:
        try:
            priced.append((float(r["Discount_Price"]), r))
        except (TypeError, ValueError):
            continue
    priced.sort(key=lambda x: x[0], reverse=True)

    print("\nTop 10 Most Expensive Products (by Discount Price):")
    print("-" * 90)
    for i, (price, r) in enumerate(priced[:10], 1):
        print(f"{i:2d}. {r['Product_Name']}")
        print(f"     Final Price : PHP {price:,.2f}")
        print(f"     URL         : {r['Product_URL']}\n")

    # ── Step 5B: Top 20 by Rating + Reviews (Review_Count > 150) ──
    eligible = []
    for r in rows:   # full set, not just main CSV
        try:
            rc = int(r["Review_Count"])
            rs = float(r["Rating_Score"])
        except (ValueError, TypeError):
            continue
        if rc > 150:
            eligible.append((rs, rc, r))
    eligible.sort(key=lambda x: (-x[0], -x[1]))

    ranked = []
    rank, prev_key = 0, None
    for i, (rs, rc, r) in enumerate(eligible, 1):
        key = (rs, rc)
        if key != prev_key:
            rank = i
            prev_key = key
        ranked.append({"Rank": rank, **r})

    top20 = ranked[:20]
    with open(TOP20_CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["Rank"] + CSV_HEADERS)
        w.writeheader()
        w.writerows(top20)
    print(f"[csv] ✅ {len(top20)} rows → {TOP20_CSV}")

print('✅ Phase 3 functions defined.')

In [ ]:
# ── Run Phase 3 ──
rows = [
    merge(c, details.get(c.get("group_key") or c["product_url"]))
    for c in cards
]
write_outputs(rows)
print("\n🎉 Scraping complete!")

---
## 📥 Download Output Files

Run the cell below to download both CSVs directly from Colab to your computer.

In [ ]:
from google.colab import files
import os

for fname in [MAIN_CSV, TOP20_CSV]:
    if os.path.exists(fname):
        files.download(fname)
        print(f"⬇️  Downloading {fname} …")
    else:
        print(f"⚠️  {fname} not found — run Phase 3 first.")

---
## 👀 Quick Preview of Results

In [ ]:
import pandas as pd

print("=" * 70)
print(f"nike_products.csv  — first 5 rows")
print("=" * 70)
if os.path.exists(MAIN_CSV):
    df_main = pd.read_csv(MAIN_CSV)
    print(f"Total rows: {len(df_main)}")
    display(df_main.head())
else:
    print("File not found — run Phase 3 first.")

print("\n" + "=" * 70)
print(f"top_20_rating_review.csv")
print("=" * 70)
if os.path.exists(TOP20_CSV):
    df_top = pd.read_csv(TOP20_CSV)
    display(df_top[["Rank", "Product_Name", "Rating_Score", "Review_Count", "Discount_Price"]])
else:
    print("File not found — run Phase 3 first.")

---

## 👨‍💻 About

| | |
|---|---|
| **Developer** | Somesh Shukla |
| **Email** | someshshukla263@gmail.com |
| **Phone** | 9179682083 |
| **Project 1** | [shikshasync.in](https://shikshasync.in) |
| **Project 2** | [dharmsetu.org.in](https://dharmsetu.org.in) |
| **Challenge** | Relu Consultancy — Data Extraction Engineer |
| **Target** | [nike.com/ph/w](https://www.nike.com/ph/w) |
| **Tech Stack** | Python · Playwright · FastAPI · Supabase · Render |